# Customer Travel Churn Prediction
Predict whether a customer will churn based on travel-related features using a Random Forest model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

## 1. Load Data

In [ ]:
# FIX: Dataset is TAB-delimited — use sep='\t'
df = pd.read_csv('Customertravel.csv', sep='\t')
print('Shape:', df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Columns:', df.columns.tolist())
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nTarget distribution:\n', df['Target'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn distribution
df['Target'].value_counts().rename({0: 'Stay', 1: 'Churn'}).plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black'
)
axes[0].set_title('Churn Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Age distribution by churn
df.groupby('Target')['Age'].plot(kind='kde', ax=axes[1], legend=True)
axes[1].set_title('Age Distribution by Churn')
axes[1].set_xlabel('Age')
axes[1].legend(['Stay (0)', 'Churn (1)'])

plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
# FIX: Removed incorrect references to 'customerID' and 'TotalCharges'
# which do not exist in this dataset.

# Fill missing numeric values (none expected, but kept as safeguard)
df.fillna(df.mean(numeric_only=True), inplace=True)

# FIX: Use astype(str) before LabelEncoding to handle pandas StringDtype
label_encoders = {}
for col in df.columns:
    if str(df[col].dtype) in ('object', 'str') or str(df[col].dtype).startswith('string'):
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f'Encoded: {col} -> classes: {le.classes_.tolist()}')

print('\nAll numeric now:\n', df.dtypes)

## 4. Train / Test Split

In [ ]:
# FIX: Target column is 'Target', not 'Churn'
X = df.drop('Target', axis=1)
y = df['Target']

print('Features:', X.columns.tolist())
print('X shape:', X.shape, '| y shape:', y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')

## 5. Train Model

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print('Model trained successfully!')

## 6. Evaluation

In [ ]:
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=['Stay', 'Churn']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stay', 'Churn'],
            yticklabels=['Stay', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
importances.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
joblib.dump(model, 'model.pkl')
print('model.pkl saved!')